# 01 - Exploratory Data Analysis (EDA)

This notebook loads all city CSV files, validates schema, computes AQI from PM2.5 using Indian breakpoints, performs exploratory visual analysis, and saves outputs for downstream phases.

Safety goals:
- Validate required files and columns before processing.
- Parse and coerce data robustly.
- Save outputs in a reproducible folder structure.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')

RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
FIG_DIR = Path('outputs/figures')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    'delhi_combined.csv',
    'mumbai_combined.csv',
    'bengaluru_combined.csv',
    'chennai_combined.csv',
    'kolkata_combined.csv',
    'hyderabad_combined.csv',
    'jaipur_combined.csv',
    'lucknow_combined.csv',
    'gwalior_combined.csv',
    'visakhapatnam_combined.csv',
]

REQUIRED_COLUMNS = ['Timestamp', 'Location', 'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3']
POLLUTANT_COLUMNS = ['PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3']

missing_files = [f for f in EXPECTED_FILES if not (RAW_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required input files: {missing_files}')

print(f'Validated {len(EXPECTED_FILES)} input files.')


## Load And Validate Input Data

What to look for:
- All files load without schema errors.
- Timestamps parse correctly with minimal invalid rows.
- Numeric pollutant fields are standardized for safe aggregation and plotting.


In [ ]:
frames = []

for file_name in EXPECTED_FILES:
    path = RAW_DIR / file_name
    df = pd.read_csv(path)
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f'{file_name} is missing required columns: {missing_cols}')

    city = file_name.replace('_combined.csv', '')
    df['City'] = city
    df = df[REQUIRED_COLUMNS + ['City']].copy()
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

# Robust parsing and coercion
combined['Timestamp'] = pd.to_datetime(combined['Timestamp'], format='%d-%m-%Y', errors='coerce')
invalid_timestamp_count = combined['Timestamp'].isna().sum()
if invalid_timestamp_count:
    print(f'Invalid Timestamp rows dropped: {invalid_timestamp_count}')
combined = combined.dropna(subset=['Timestamp']).copy()

for col in POLLUTANT_COLUMNS:
    combined[col] = pd.to_numeric(combined[col], errors='coerce')

combined = combined.sort_values(['City', 'Timestamp']).reset_index(drop=True)

print(f'Combined shape after parsing: {combined.shape}')
combined.head()


In [ ]:
print('Overall dtypes:')
display(combined.dtypes.to_frame('dtype'))

summary_rows = []
for city, grp in combined.groupby('City', sort=True):
    row = {'City': city, 'rows': len(grp), 'cols': grp.shape[1]}
    for col in combined.columns:
        row[f'null_{col}'] = int(grp[col].isna().sum())
    summary_rows.append(row)

city_summary = pd.DataFrame(summary_rows).sort_values('City').reset_index(drop=True)
display(city_summary)

null_percent = (
    combined.groupby('City')[REQUIRED_COLUMNS]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index()
)
display(null_percent)


## Compute AQI From PM2.5

Rules implemented:
- PM2.5 values below 0 are clamped to 0.
- PM2.5 values above 500 are clamped to 500.
- Indian AQI bands are applied with linear interpolation.
- Final AQI is clipped to [0, 500].


In [ ]:
AQI_BREAKPOINTS = [
    (0, 30, 0, 50),
    (30, 60, 51, 100),
    (60, 90, 101, 200),
    (90, 120, 201, 300),
    (120, 250, 301, 400),
    (250, 500, 401, 500),
]

def pm25_to_aqi(pm):
    if pd.isna(pm):
        return np.nan
    pm = float(np.clip(pm, 0, 500))
    for bp_low, bp_high, aqi_low, aqi_high in AQI_BREAKPOINTS:
        if bp_low <= pm <= bp_high:
            if bp_high == bp_low:
                return float(np.clip(aqi_high, 0, 500))
            aqi = ((aqi_high - aqi_low) / (bp_high - bp_low)) * (pm - bp_low) + aqi_low
            return float(np.clip(aqi, 0, 500))
    return 500.0

combined['AQI'] = combined['PM2.5'].apply(pm25_to_aqi)
combined[['PM2.5', 'AQI']].head()


In [ ]:
city_order = sorted(combined['City'].dropna().unique().tolist())
palette = dict(zip(city_order, sns.color_palette('tab10', n_colors=len(city_order))))

combined['Month'] = combined['Timestamp'].dt.to_period('M').dt.to_timestamp()


## Plot 1 - Monthly Average PM2.5 (All Cities)

What to look for:
- Seasonal PM2.5 swings over time.
- Relative pollution levels between cities across months.


In [ ]:
monthly_pm25 = (
    combined.groupby(['Month', 'City'], as_index=False)['PM2.5']
    .mean()
)

plt.figure(figsize=(14, 6))
for city in city_order:
    city_df = monthly_pm25[monthly_pm25['City'] == city]
    plt.plot(city_df['Month'], city_df['PM2.5'], label=city, color=palette[city], linewidth=1.8)

plt.title('Monthly Average PM2.5 Across Cities')
plt.xlabel('Month')
plt.ylabel('PM2.5')
plt.legend(loc='upper right', ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'monthly_pm25_all_cities.png', dpi=200)
plt.close()


## Plot 2 - AQI Distribution Per City (Box Plot)

What to look for:
- Median and spread of AQI by city.
- Cities with heavier tails or more frequent extreme values.


In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=combined, x='City', y='AQI', hue='City', order=city_order, palette=palette, dodge=False, legend=False)
plt.title('AQI Distribution Per City')
plt.xlabel('City')
plt.ylabel('AQI')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'aqi_distribution_per_city.png', dpi=200)
plt.close()


## Plot 3 - Missing Value Percentage Heatmap

What to look for:
- Which cities and variables carry the highest missingness.
- Whether missingness is concentrated in specific pollutants.


In [ ]:
missing_pct = combined.groupby('City')[REQUIRED_COLUMNS].apply(lambda x: x.isna().mean() * 100)

plt.figure(figsize=(12, 8))
sns.heatmap(missing_pct, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Missing (%)'})
plt.title('Missing Value Percentage By City And Column')
plt.xlabel('Column')
plt.ylabel('City')
plt.tight_layout()
plt.savefig(FIG_DIR / 'missing_values_heatmap.png', dpi=200)
plt.close()


## Plot 4 - Average PM2.5 Per City (Descending)

What to look for:
- Relative pollution burden by city.
- City ranking based on mean PM2.5.


In [ ]:
avg_pm25_city = combined.groupby('City', as_index=False)['PM2.5'].mean().sort_values('PM2.5', ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=avg_pm25_city, x='City', y='PM2.5', hue='City', palette=palette, dodge=False, legend=False)
plt.title('Average PM2.5 Per City (Descending)')
plt.xlabel('City')
plt.ylabel('Average PM2.5')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'avg_pm25_per_city.png', dpi=200)
plt.close()


## Plot 5 - Delhi PM2.5 Trend With Threshold

What to look for:
- Long-term PM2.5 behavior in Delhi.
- Frequency and severity of exceedances over PM2.5 = 60 threshold.


In [ ]:
delhi_df = combined[combined['City'] == 'delhi'].copy()

plt.figure(figsize=(14, 6))
plt.plot(delhi_df['Timestamp'], delhi_df['PM2.5'], color=palette['delhi'], linewidth=1.5)
plt.axhline(60, color='red', linestyle='--', linewidth=1.5, label='PM2.5 = 60 threshold')
plt.title('Delhi PM2.5 Trend')
plt.xlabel('Date')
plt.ylabel('PM2.5')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'delhi_pm25_trend.png', dpi=200)
plt.close()


## Plot 6 - Pollutant Correlation Heatmap

What to look for:
- Linear relationships between pollutants.
- Features that may carry overlapping information for modeling.


In [ ]:
corr = combined[POLLUTANT_COLUMNS].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap Of Pollutants')
plt.tight_layout()
plt.savefig(FIG_DIR / 'pollutant_correlation.png', dpi=200)
plt.close()


## Key Observations

The following bullets are computed from this dataset at runtime.

In [ ]:
highest_avg_pm25 = avg_pm25_city.iloc[0]
lowest_avg_pm25 = avg_pm25_city.iloc[-1]

city_aqi_mean = combined.groupby('City')['AQI'].mean().sort_values(ascending=False)
top_aqi_city = city_aqi_mean.index[0]
top_aqi_val = city_aqi_mean.iloc[0]

most_missing_city = missing_pct.mean(axis=1).sort_values(ascending=False).index[0]
most_missing_val = missing_pct.mean(axis=1).max()

delhi_exceedance = (delhi_df['PM2.5'] > 60).mean() * 100

top_corr = corr.where(~np.eye(corr.shape[0], dtype=bool)).stack().sort_values(ascending=False)
top_pair = top_corr.index[0]
top_pair_corr = top_corr.iloc[0]

obs_md = f"""
- Highest average PM2.5 city: **{highest_avg_pm25['City']}** ({highest_avg_pm25['PM2.5']:.2f}), while the lowest is **{lowest_avg_pm25['City']}** ({lowest_avg_pm25['PM2.5']:.2f}).
- Highest mean AQI is in **{top_aqi_city}** at **{top_aqi_val:.2f}**.
- The city with highest average missingness across required columns is **{most_missing_city}** at **{most_missing_val:.2f}%**.
- In Delhi, PM2.5 exceeds the threshold (60) on approximately **{delhi_exceedance:.1f}%** of recorded days.
- Strongest positive pollutant correlation is between **{top_pair[0]}** and **{top_pair[1]}** (**r = {top_pair_corr:.2f}**).
"""

display(Markdown(obs_md))


## Save Processed Output

Save merged city-level dataset with computed AQI for downstream preprocessing.

In [ ]:
output_path = PROCESSED_DIR / 'combined_all_cities.csv'
combined.to_csv(output_path, index=False)
print(f'Saved processed dataset to: {output_path}')
